## Анализ вероятности ухода клиента из банка

In [ ]:
import time
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from catboost import CatBoostClassifier
from

RANDOM_STATE = 42

/Users/georgiy/.pyenv/versions/3.12.8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [18]:
X, y = train.drop(columns=['id', 'Churn']), train['Churn'].map({'No': 0, 'Yes': 1})

In [19]:
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')
X = preprocessor.fit_transform(X)
X.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,-0.358885,-0.302342,-0.185604,-0.357076,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,-0.358885,0.854793,0.116964,0.545399,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,-0.358885,0.854793,1.111575,1.421875,0.0,1.0,0.0,1.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,-0.358885,-1.419575,0.123402,-1.029637,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,-0.358885,-1.419575,0.147543,-1.029743,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


### Создание Baseline

In [ ]:
class modelsHub:
    def __init__(self):
        self.results = []
        self.models = {}

    def _get_scores(self, model, X):
        if hasattr(model, 'predict_proba'):
            return model.predict_proba(X)[:, 1]
        if hasattr(model, 'decision_function'):
            return model.decision_function(X)
        return None

    def fit_predict(self, model, X_train, y_train, X_valid, y_valid, model_name, choice_best=False):
        fit_start = time.perf_counter()
        model.fit(X_train, y_train)
        fit_time = time.perf_counter() - fit_start

        pred_start = time.perf_counter()
        y_pred = model.predict(X_valid)
        pred_time = time.perf_counter() - pred_start

        y_score = self._get_scores(model, X_valid)

        row = {
            'model': model_name,
            'fit_time_sec': fit_time,
            'predict_time_sec': pred_time,
            'accuracy': accuracy_score(y_valid, y_pred),
            'precision': precision_score(y_valid, y_pred, zero_division=0),
            'recall': recall_score(y_valid, y_pred, zero_division=0),
            'f1': f1_score(y_valid, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_valid, y_score) if y_score is not None else np.nan,
        }

        if choice_best:
            existing_indices = [i for i, result in enumerate(self.results) if result['model'] == model_name]
            if existing_indices:
                best_existing_index = max(existing_indices, key=lambda i: self.results[i]['roc_auc'])
                if row['roc_auc'] <= self.results[best_existing_index]['roc_auc']:
                    return self.results[best_existing_index]
                self.results = [result for i, result in enumerate(self.results) if i not in existing_indices]

        self.results.append(row)
        self.models[model_name] = model
        return row

    def leaderboard(self):
        if not self.results:
            return pd.DataFrame()

        df = pd.DataFrame(self.results).sort_values('roc_auc', ascending=False).reset_index(drop=True)

        def highlight_best_roc_auc(row):
            best_roc_auc = df['roc_auc'].max()
            if pd.notna(row['roc_auc']) and row['roc_auc'] == best_roc_auc:
                return ['background-color: #F0FFF0'] * len(row)
            return [''] * len(row)

        return (
            df.style
            .format({
                'fit_time_sec': '{:.4f}',
                'predict_time_sec': '{:.4f}',
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
            })
            .apply(highlight_best_roc_auc, axis=1)
        )

In [6]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

hub = modelsHub()
hub.fit_predict(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    X_train,
    y_train,
    X_valid,
    y_valid,
    model_name='LogisticRegression',
)
hub.leaderboard()

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc
0,LogisticRegression,0.5625,0.0111,0.8563,0.6879,0.6623,0.6749,0.9085


### RandomForest

### CatBoost

In [ ]:

depth = [4, 6, 8]
iterations = [100, 1000, 5000]
reg_lambdas = [0.1, 0.5]

best_params = {
    'depth': 0,
    'iterations': 0,
    'reg_lambda': 0,
}

best_roc_auc = 0
param_grid = list(product(depth, iterations, reg_lambdas))
for d, it, reg in tqdm(param_grid, total=len(param_grid), desc='CatBoost tuning'):
    model = CatBoostClassifier(depth=d, iterations=it, reg_lambda=reg, random_state=RANDOM_STATE, verbose=0)
    row = hub.fit_predict(
        model,
        X_train,
        y_train,
        X_valid,
        y_valid,
        model_name=f'CatBoostClassifier',
        choice_best=True,
    )
    if row['roc_auc'] > best_roc_auc:
        best_roc_auc = row['roc_auc']
        best_params = {
            'depth': d,
            'iterations': it,
            'reg_lambda': reg,
        }

print(best_params)

CatBoost tuning: 100%|██████████| 18/18 [09:58<00:00, 33.25s/it]

{'depth': 4, 'iterations': 5000, 'reg_lambda': 0.5}


In [ ]:
hub.leaderboard()


,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc
0,CatBoostClassifier,62.1441,0.1269,0.8627,0.7162,0.6469,0.6798,0.9169
1,LogisticRegression,0.5625,0.0111,0.8563,0.6879,0.6623,0.6749,0.9085


In [ ]:
model = CatBoostClassifier(**best_params, random_state=RANDOM_STATE, verbose=0)

